# Table of Contents
- [RAG with LangChain](#RAG-with-LangChain)
- [RAG with LangGraph](#RAG-with-LangGraph)

# RAG with LangChain
### When Chains Are Fine
* For simple linear RAG (retrieve → generate → done), chains work perfectly well. But LangGraph gives you room to grow without refactoring - you can add additional nodes easily or introduce cycles (see below)
* The following is a condensed version of `https://github.com/agnedil/rag-demo-with-streamlit/blob/main/advanced_rag_history.py`

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from langchain_community.llms import Replicate
from langchain_community.document_loaders import OnlinePDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import CohereEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank
from langchain.chains import ConversationalRetrievalChain

def create_rag_chain(pdf_links, chunk_size=1500, top_k=5):
    """Create an advanced RAG system with hybrid retrieval and reranking."""
    # Load and chunk documents
    docs = [OnlinePDFLoader(url).load()[0] for url in pdf_links]
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    
    # Create retrievers
    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = top_k
    faiss = FAISS.from_documents(chunks, CohereEmbeddings(model="embed-english-light-v3.0")).as_retriever(search_kwargs={"k": top_k})
    
    # Combine with ensemble and reranker
    ensemble = EnsembleRetriever(retrievers=[bm25, faiss], weights=[0.6, 0.4])
    retriever = ContextualCompressionRetriever(base_retriever=ensemble, base_compressor=CohereRerank(top_n=5))
    
    # Create chain
    llm = Replicate(model='meta/llama-2-70b-chat', model_kwargs={"temperature": 0.5, "top_p": 1, "max_new_tokens": 1000})
    return ConversationalRetrievalChain.from_llm(llm, retriever, return_source_documents=True)

def query_with_history(chain, query, history=[]):
    """Query the chain and maintain 5 last utterances in chat history"""
    result = chain({"question": query, "chat_history": history})
    history.append((query, result["answer"]))
    return result['answer'], history[-5:]

# Usage
chain = create_rag_chain(['url1.pdf', 'url2.pdf'])
history = []
answer, history = query_with_history(chain, "What is...?", history)

# RAG with LangGraph
## Why LangGraph

### 1. Explicit Control Flow
* LangGraph: You can see exactly how data flows between steps (retrieve → generate) - makes debugging and understanding the pipeline much easier.
* LangChain chains: Control flow is more implicit and hidden in chain internals 

### 2. State Management
* LangGraph: Explicit state object (RAGState) that you control
* Chains: State is passed implicitly, harder to inspect or modify mid-pipeline

### 3. Scalability for Complex Workflows
* LangGraph gives you room to grow without refactoring. The code below is simple now, but LangGraph makes it trivial to add query rewriting, query routing, iterative retrieval, response evaluation, reranking, self-critique, multi-hop retrieval or other agentic behaviors later: just use `g.add_node()`

In [ ]:
# INDEX.PY - load, chunk, and vectorize
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

DOCS_DIR = Path("docs")
def load_docs():
    return [ doc for fp in DOCS_DIR.glob("*") if fp.is_file()
             for doc in TextLoader(str(fp), encoding="utf-8").load() ]

splitter   = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120,)
chunks     = splitter.split_documents(load_docs())
embeddings = OpenAIEmbeddings()

Chroma.from_documents( documents=chunks, embedding=embeddings,
                       persist_directory="chroma_db", collection_name="rag_docs", )
print("✅ Index built")

In [ ]:
# RAG_GRAPH.PY
from typing import TypedDict, List
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str
    docs: List[Document]
    answer: str
        
# load a pre-indexed vector DB
vectorstore = Chroma( persist_directory="chroma_db",
                      collection_name="rag_docs",
                      embedding_function=OpenAIEmbeddings(), )

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
def retrieve(state: RAGState) -> RAGState:
    q = state["question"]
    docs = retriever.invoke(q)
    return {"question": q, "docs": docs, "answer": ""}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
def generate(state: RAGState) -> RAGState:
    q = state["question"]
    docs = state["docs"]
    context = "\n\n".join([d.page_content for d in docs])
    messages = [ SystemMessage( content=(
                "Answer using ONLY the context. If missing is missing,"
                "say you don't have that info in the documents." ) ),
                HumanMessage(content=f"Question:\n{q}\n\nContext:\n{context}") ]
    resp = llm.invoke(messages)
    return {**state, "answer": resp.content}

def build_graph():
    g = StateGraph(RAGState)
    g.add_node("retrieve", retrieve)
    g.add_node("generate", generate)

    g.add_edge(START, "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

rag_app = build_graph()

In [ ]:
# RUN.PY
from rag_graph import rag_app

while True:
    q = input("\nAsk: ").strip()
    if not q: break
    result = rag_app.invoke( {"question": q, "docs": [], "answer": ""} )
    print("\nAnswer:\n", result["answer"])
    print("\nChunks used:", len(result["docs"]))

In [ ]:
# NOTE: With LangGraph, it is easy to insert nodes and rewire
g.add_node("rewrite_query", rewrite)
g.add_node("rerank", rerank)
g.add_node("evaluate_response", evaluate)